# 🧩 YouTube Comment Scraper (Multiple Channels + Multiple API Keys)

This notebook extracts **comments from all videos of multiple YouTube channels** using:
- **yt-dlp** → to get all video IDs from a channel  
- **YouTube Data API (v3)** → to fetch comments  
- **Multiple API keys** (switches when one exceeds quota)  
- **Separate text files per channel**

We'll proceed step-by-step:
1. Import dependencies  
2. Configure API keys & channels  
3. Fetch video IDs  
4. Scrape comments with auto key rotation  
5. Save results in channel-wise files


In [10]:
import time
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from tqdm import tqdm
from yt_dlp import YoutubeDL
import os


### ⚙️ Configuration

Here, we define:
- **`API_KEYS`**: a list of YouTube Data API keys (the script will switch when quota is exceeded)  
- **`CHANNEL_URLS`**: list of channel URLs (must be in `/@channelName/videos` format)  
- **Other parameters**: retry limits, sleep times, etc.


In [ ]:


CHANNEL_URLS = [
    "https://www.youtube.com/@TheNepaliComment/videos",
    "https://www.youtube.com/@ProjectKura/videos",
    "https://www.youtube.com/@IdeapreneurNepal/videos",
    "https://www.youtube.com/@MusicNepalPvtLtd/videos",
    "https://www.youtube.com/@4KGamingNepal/videos",
    "https://www.youtube.com/@OSRDigital/videos",
]

MAX_RETRIES = 3
SLEEP_BETWEEN_VIDEOS = 0.2 


### 🧱 Build YouTube API Client

This helper function returns a YouTube Data API service object
for the provided API key.


In [4]:
def get_youtube_service(api_key):
    """Initialize YouTube API client."""
    return build("youtube", "v3", developerKey=api_key)


### 🎥 Fetch All Video IDs of a Channel

We use `yt_dlp` to extract **video IDs** from a YouTube channel without downloading the videos.

- Works with any `@channelName/videos` URL.
- Returns a list of video IDs.


In [12]:
def get_channel_video_ids(channel_url):
    """Fetch all video IDs from the given YouTube channel URL."""
    ydl_opts = {"extract_flat": True, "quiet": True}
    video_ids = []
    with YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(channel_url, download=False)
        for entry in info.get("entries", []):
            if entry.get("id"):
                video_ids.append(entry["id"])
    return video_ids


### 🧾 Helper: Detect Devanagari Characters

This function checks if a comment contains Devanagari (Nepali/Hindi script).
We'll skip such comments if you only want English/Roman text.


In [13]:
def contains_devanagari(text):
    """Detects if the text contains Devanagari script."""
    return any('\u0900' <= char <= '\u097F' for char in text)


### 💬 Scrape Comments from a Single Video

This function:
- Fetches all comments for a given video ID
- Handles pagination (`nextPageToken`)
- Handles quota exhaustion (switches API key)
- Retries on rate-limit or temporary server errors


In [14]:
def scrape_comments_from_video(youtube, video_id, api_keys):
    comments = []
    next_page_token = None
    retries = 0
    key_index = 0

    print(f"\n🔍 Fetching comments for video ID: {video_id}")

    while True:
        try:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                pageToken=next_page_token,
                maxResults=100,
                textFormat="plainText",
                fields="items/snippet/topLevelComment/snippet/textDisplay,nextPageToken"
            ).execute()

            for item in response.get("items", []):
                text = item["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
                comments.append(text)

            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break

        except HttpError as e:
            if e.resp.status == 403 and "quota" in str(e).lower():
                print("❌ QUOTA EXCEEDED for current API key.")
                key_index += 1
                if key_index < len(api_keys):
                    print("🔁 Switching to next API key...")
                    youtube = get_youtube_service(api_keys[key_index])
                    continue
                else:
                    print("🚫 All API keys exhausted. Stopping this video.")
                    break

            elif e.resp.status in [429, 500, 503]:
                if retries < MAX_RETRIES:
                    wait_time = 2 ** retries
                    print(f"⚠️ Rate limited. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    retries += 1
                    continue
                else:
                    print("❌ Max retries reached. Skipping video.")
                    break
            else:
                print(f"❌ Error fetching comments: {e}")
                break
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            break

    print(f"✅ Collected {len(comments)} comments from video {video_id}")
    return comments


### 📁 Process All Channels

This main function:
1. Loops over each channel  
2. Fetches all video IDs  
3. For each video:
   - Fetches comments  
   - Saves them to a text file named after the channel  
4. Creates separate files per channel, e.g.:


In [15]:
def main():
    key_index = 0
    youtube = get_youtube_service(API_KEYS[key_index])

    for channel_url in CHANNEL_URLS:
        print(f"\n============================")
        print(f"📺 Processing Channel: {channel_url}")
        print(f"============================")

        # Extract clean channel name
        channel_name = channel_url.split("@")[-1].split("/")[0]
        output_file = f"{channel_name}_comments.txt"

        try:
            video_ids = get_channel_video_ids(channel_url)
            print(f"🎬 Found {len(video_ids)} videos for {channel_name}")
        except Exception as e:
            print(f"❌ Failed to get videos for {channel_name}: {e}")
            continue

        with tqdm(total=len(video_ids), desc=f"Processing {channel_name}", unit="video") as pbar:
            for video_id in video_ids:
                comments = scrape_comments_from_video(youtube, video_id, API_KEYS)
                pbar.update(1)

                try:
                    with open(output_file, "a", encoding="utf-8") as f:
                        for comment in comments:
                            if not contains_devanagari(comment):
                                f.write(comment.strip() + "\n")
                    print(f"💾 Saved {len(comments)} comments to {output_file}")
                except Exception as e:
                    print(f"❌ Failed to write file for {channel_name}: {e}")

                time.sleep(SLEEP_BETWEEN_VIDEOS)

        print(f"✅ Finished scraping channel: {channel_name}\n")


### ▶️ Run the Pipeline

Finally, call `main()` to execute the full scraping process.

> This will:
> - Rotate API keys if quota exceeds
> - Process all listed channels
> - Save comments to separate `.txt` files


In [16]:
if __name__ == "__main__":
    main()



📺 Processing Channel: https://www.youtube.com/@TechLekhYT/videos
🎬 Found 666 videos for TechLekhYT


Processing TechLekhYT:   0%|                         | 0/666 [00:00<?, ?video/s]


🔍 Fetching comments for video ID: 8wYPxIrorV0


Processing TechLekhYT:   0%|                 | 1/666 [00:00<04:40,  2.37video/s]

✅ Collected 17 comments from video 8wYPxIrorV0
💾 Saved 17 comments to TechLekhYT_comments.txt

🔍 Fetching comments for video ID: bs1TuJhJ93k


...

✅ Collected 17 comments from video ycvGYDPBLrI
💾 Saved 17 comments to GadgetByteNepali_comments.txt

🔍 Fetching comments for video ID: OUcF4GgIxHk


Processing GadgetByteNepali: 100%|█████████| 635/635 [08:37<00:00,  2.01video/s]

✅ Collected 65 comments from video OUcF4GgIxHk
💾 Saved 65 comments to GadgetByteNepali_comments.txt


Processing GadgetByteNepali: 100%|█████████| 635/635 [08:38<00:00,  1.23video/s]

✅ Finished scraping channel: GadgetByteNepali

